# Laboratoriya işi 4 – Tapşırıq 14
**Mövzu:** Çoxqatlı perceptron (MLP) modellərinin qurulması, aktivləşmə funksiyaları, backpropagation ilə öyrədilməsi

**Tapşırıq:** Ev qiymətlərini çoxdəyişənli MLP reqressiya modeli ilə proqnozlaşdırın.

**Plan:** data yüklə → train/test böl → standartlaşdır → MLP qur və öyrət → qiymətləndir → aktivləşmə funksiyalarını müqayisə et

## 1. Kitabxanaların yüklənməsi

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 2. Datanın yüklənməsi
California ev qiymətləri datasetindən istifadə edirik. Hər sətir bir rayonu göstərir.
Hədəf dəyişən **MedHouseVal** – evlərin median qiymətidir (vahid: 100 000 $).

In [ ]:
data = fetch_california_housing(as_frame=True)
df = data.frame

print("Data ölçüsü:", df.shape)
df.head()

In [ ]:
# Boş (itkin) dəyərlər və əsas statistika
print("Boş dəyərlərin sayı:", df.isnull().sum().sum())
df.describe()

## 3. Giriş (X) və hədəf (y) ayrılması, train/test bölünməsi
Datanın 80%-i modeli öyrətmək, 20%-i yoxlamaq üçündür.

In [ ]:
X = df.drop(columns="MedHouseVal")   # 8 giriş dəyişəni
y = df["MedHouseVal"]                # hədəf: ev qiyməti

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Train:", X_train.shape, " Test:", X_test.shape)

## 4. Standartlaşdırma
MLP dəyişənlərin miqyasına həssasdır, ona görə bütün giriş dəyişənlərini orta=0, standart sapma=1 edirik.
Scaler yalnız **train** üzərində öyrənilir (test datasının məlumatı "sızmasın" deyə).

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

## 5. MLP modelinin qurulması və öyrədilməsi
- **2 gizli qat:** 64 və 32 neyron
- **Aktivləşmə funksiyası:** ReLU
- **Öyrətmə:** backpropagation + Adam optimizatoru (itki funksiyası: kvadratik xəta)
- `early_stopping=True` – nəticə yaxşılaşmayanda öyrətmə avtomatik dayanır

In [ ]:
model = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    max_iter=300,
    early_stopping=True,
    random_state=42,
)
model.fit(X_train_s, y_train)

print("Epoxa sayı:", model.n_iter_)
print("Qat sayı (giriş+gizli+çıxış):", model.n_layers_)

### Öyrənmə əyrisi
Hər epoxada backpropagation ilə xəta azalır – əyri aşağı enirsə, model öyrənir.

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(model.loss_curve_)
plt.xlabel("Epoxa")
plt.ylabel("Xəta (loss)")
plt.title("Öyrətmə zamanı xətanın azalması")
plt.grid(True)
plt.show()

## 6. Modelin qiymətləndirilməsi
- **MAE** – orta mütləq xəta
- **RMSE** – kök orta kvadratik xəta
- **R²** – modelin dəyişkənliyi nə qədər izah etdiyi (1-ə yaxın olması yaxşıdır)

In [ ]:
def qiymetlendir(model, X, y):
    pred = model.predict(X)
    mae = mean_absolute_error(y, pred)
    rmse = np.sqrt(mean_squared_error(y, pred))
    r2 = r2_score(y, pred)
    return mae, rmse, r2

netice = pd.DataFrame(
    [qiymetlendir(model, X_train_s, y_train), qiymetlendir(model, X_test_s, y_test)],
    columns=["MAE", "RMSE", "R2"],
    index=["Train", "Test"],
).round(3)
netice

### Real və proqnoz qiymətlərin müqayisəsi
Nöqtələr qırmızı xəttə nə qədər yaxındırsa, proqnoz o qədər dəqiqdir.

In [ ]:
y_pred = model.predict(X_test_s)

plt.figure(figsize=(5, 5))
plt.scatter(y_test, y_pred, alpha=0.3)
plt.plot([y.min(), y.max()], [y.min(), y.max()], color="red")
plt.xlabel("Real qiymət")
plt.ylabel("Proqnoz qiymət")
plt.title("Real vs Proqnoz (test)")
plt.show()

## 7. Aktivləşmə funksiyalarının müqayisəsi
Eyni modeli üç fərqli aktivləşmə funksiyası ilə öyrədib test nəticələrini müqayisə edirik:
**ReLU**, **tanh**, **sigmoid** (sklearn-də `logistic`).

In [ ]:
nəticələr = []

for akt in ["relu", "tanh", "logistic"]:
    m = MLPRegressor(
        hidden_layer_sizes=(64, 32),
        activation=akt,
        max_iter=300,
        early_stopping=True,
        random_state=42,
    )
    m.fit(X_train_s, y_train)
    mae, rmse, r2 = qiymetlendir(m, X_test_s, y_test)
    nəticələr.append([akt, mae, rmse, r2, m.n_iter_])

muqayise = pd.DataFrame(
    nəticələr, columns=["Aktivləşmə", "MAE", "RMSE", "R2", "Epoxa"]
).round(3)
muqayise

## 8. Nəticə

In [ ]:
en_yaxsi = muqayise.sort_values("R2", ascending=False).iloc[0]
print(f"Əsas model (ReLU) test R2 = {netice.loc['Test', 'R2']}, test MAE = {netice.loc['Test', 'MAE']}")
print(f"Ən yaxşı aktivləşmə funksiyası: {en_yaxsi['Aktivləşmə']} (R2 = {en_yaxsi['R2']})")

**Nəticə:**
- Çoxdəyişənli (8 giriş) MLP reqressiya modeli ilə ev qiymətləri proqnozlaşdırıldı.
- Dəyişənlər standartlaşdırıldıqdan sonra model backpropagation vasitəsilə öyrədildi; öyrənmə əyrisində xətanın azaldığı görünür.
- Train və test nəticələri bir-birinə yaxındırsa, model həddindən artıq öyrənməyib (overfitting yoxdur).
- Aktivləşmə funksiyalarının müqayisəsi göstərdi ki, funksiyanın seçimi nəticəyə təsir edir; ən yaxşı funksiya yuxarıdakı cədvəldə göstərilib.